In [ ]:
!pip install pgmpy

In [11]:
import numpy as np
import pandas as pd
from pgmpy.models import BayesianModel
from pgmpy.estimators import MaximumLikelihoodEstimator
from scipy.stats import chi2_contingency
from pgmpy.inference import VariableElimination

def build_bayesian_network_sample(data, sample_size=50, threshold=0.05):
    if sample_size is not None:
        sampled_data = data.sample(n=min(sample_size, len(data)), random_state=42)  # Sample the data.
    else:
        sampled_data = data

    variable_order = list(sampled_data.columns)
    model = BayesianModel()

    for variable in variable_order:
        model.add_node(variable)
        parents = []
        for potential_parent in model.nodes():
            if potential_parent != variable:
                contingency_table = pd.crosstab(sampled_data[potential_parent], sampled_data[variable])
                chi2, p, dof, expected = chi2_contingency(contingency_table)
                if p < threshold:
                    parents.append(potential_parent)
        model.add_edges_from([(parent, variable) for parent in parents])

    model.fit(sampled_data, estimator=MaximumLikelihoodEstimator)
    return model

# Load the heart disease dataset from the CSV file 'heart.csv'.
data = pd.read_csv('heart.csv')

# Build the Bayesian network with a sample of 50 rows.
model = build_bayesian_network_sample(data, sample_size=50)

# Print the CPDs.
for cpd in model.get_cpds():
    print("CPD for:", cpd.variable)
    print(cpd)
    print("\n")

# Append the inference code.
HeartDisease_infer = VariableElimination(model)

# Corrected query with sex:1
q = HeartDisease_infer.query(variables=['target'], evidence={'age': 37, 'sex': 1})
print(q)

CPD for: age
+---------+------+
| age(34) | 0.02 |
+---------+------+
| age(40) | 0.06 |
+---------+------+
| age(41) | 0.02 |
+---------+------+
| age(43) | 0.02 |
+---------+------+
| age(44) | 0.06 |
+---------+------+
| age(45) | 0.04 |
+---------+------+
| age(46) | 0.06 |
+---------+------+
| age(48) | 0.02 |
+---------+------+
| age(50) | 0.02 |
+---------+------+
| age(51) | 0.04 |
+---------+------+
| age(52) | 0.08 |
+---------+------+
| age(54) | 0.06 |
+---------+------+
| age(55) | 0.02 |
+---------+------+
| age(56) | 0.06 |
+---------+------+
| age(57) | 0.1  |
+---------+------+
| age(58) | 0.02 |
+---------+------+
| age(59) | 0.08 |
+---------+------+
| age(61) | 0.04 |
+---------+------+
| age(62) | 0.04 |
+---------+------+
| age(63) | 0.02 |
+---------+------+
| age(64) | 0.04 |
+---------+------+
| age(66) | 0.02 |
+---------+------+
| age(67) | 0.02 |
+---------+------+
| age(71) | 0.04 |
+---------+------+


CPD for: sex
+--------+---------+---------+---------+-